In [1]:
import sys; sys.path.append('../3rdparty/ElasticKnots/3rdparty/ElasticRods/python')
import sys; sys.path.append('../3rdparty/ElasticKnots/python')
import elastic_rods, elastic_knots
import numpy as np, matplotlib.pyplot as plt, time, io, os
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh

from helpers import *
from parametric_curves import *
import py_newton_optimizer 

from linkage_vis import LinkageViewer as Viewer, CenterlineViewer
from tri_mesh_viewer import PointCloudViewer, PointCloudMesh

%load_ext autoreload
%autoreload 2

import parallelism
parallelism.set_max_num_tbb_threads(1)

from MEP import MEP

Failed to load offscreen viewer: Could not load compiled module; is OffscreenRenderer missing a dependency?


In [2]:
knot_name = '5_2/0001.obj'
file = '../data/L400-r0.2-UpTo9Crossings/' + knot_name
rod_radius = 0.2
material = elastic_rods.RodMaterial('ellipse', 2000, 1, [rod_radius, rod_radius])
centerline = read_nodes_from_file(file)  # supported formats: obj, txt
pr = define_periodic_rod(centerline, material)
rod_list = elastic_knots.PeriodicRodList([pr])
len(rod_list.getDoFs())

1601

In [3]:
view = Viewer(rod_list, width=1024, height=800)
view.show()


/home/simon/miniconda3/envs/ElasticKnots/lib/python3.9/site-packages/jupyter_client/session.py:721: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


Renderer(camera=PerspectiveCamera(aspect=1.28, children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0),…

In [ ]:
def callback(problem, iteration):
    if iteration % 5 == 0:
        view.update()
for i in range(100,1000):
    print(f"iterration: {i}")
    optimizerOptions = py_newton_optimizer.NewtonOptimizerOptions()
    optimizerOptions.niter = 10000
    optimizerOptions.gradTol = 1e-8
    hessianShift = 1e-4 * compute_min_eigenval_straight_rod(pr)

    problemOptions = elastic_knots.ContactProblemOptions()
    problemOptions.contactStiffness = 1e+3
    problemOptions.dHat = 2*rod_radius * 0.01 *i
    fixedVars = []   
    
    report = elastic_knots.compute_equilibrium(
        rod_list, problemOptions, optimizerOptions, 
        fixedVars=fixedVars,
        externalForces=np.zeros(rod_list.numDoF()),
        softConstraints=[],
        callback=callback,
        hessianShift=hessianShift
        )
    view.update()

iterration: 100
0	0.909797	0.0197564	0.0197564	1	1
1	0.909744	2.82356e-06	2.82356e-06	1	1
2	0.909744	8.3962e-10	8.3962e-10	1	1
3	0.909744	8.60099e-11	8.60099e-11	1	0
4	0.909744	1.18901e-11	1.18901e-11	1	0
iterration: 101
0	0.914298	1.39206	1.39206	1	1
1	0.910511	0.365272	0.365272	1	1
2	0.909948	0.0991779	0.0991779	1	1
3	0.909856	0.025084	0.025084	1	1
4	0.909843	0.00520853	0.00520853	1	1
5	0.909839	0.00118926	0.00118926	1	0
6	0.909833	0.000836364	0.000836364	1	0
7	0.909833	5.47701e-05	5.47701e-05	1	0
8	0.909833	1.53959e-06	1.53959e-06	1	0
9	0.909833	1.87669e-09	1.87669e-09	1	0
iterration: 102
0	0.914292	1.32057	1.32057	1	1
1	0.910583	0.345161	0.345161	1	1
2	0.910037	0.0950891	0.0950891	1	1
3	0.909946	0.0241911	0.0241911	1	1
4	0.909932	0.00480734	0.00480734	1	1
5	0.909928	0.00118237	0.00118237	1	0
6	0.909922	0.000805272	0.000805272	1	0
7	0.909922	9.75401e-05	9.75401e-05	1	0
8	0.909922	2.0427e-06	2.0427e-06	1	0
9	0.909922	1.27888e-09	1.27888e-09	1	0
iterration: 103
0	0.914379	1.29623	1.29

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	0.934263	3.6566	3.6566	1	1
1	0.923725	0.939809	0.939809	1	1
2	0.92229	0.246757	0.246757	1	1
3	0.922081	0.0640356	0.0640356	1	1
4	0.922048	0.015884	0.015884	1	1
5	0.922043	0.00326809	0.00326809	1	1
6	0.922041	0.000742662	0.000742662	0.125	0
7	0.922041	0.00176954	0.00176954	1	0
8	0.922038	0.000541989	0.000541989	0.25	0
9	0.922038	0.000388529	0.000388529	1	0
10	0.922038	0.000141448	0.000141448	1	0
11	0.922038	6.42038e-06	6.42038e-06	1	0
12	0.922038	3.33317e-07	3.33317e-07	1	0
13	0.922038	2.49207e-11	2.49207e-11	1	0
iterration: 251


The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	0.934404	3.66184	3.66184	1	1
1	0.9238	0.940116	0.940116	1	1
2	0.922366	0.245787	0.245787	1	1
3	0.922159	0.0637635	0.0637635	1	1
4	0.922127	0.0158577	0.0158577	1	1
5	0.922121	0.00326756	0.00326756	1	1
6	0.92212	0.000726347	0.000726347	0.0625	0
7	0.922119	0.00108178	0.00108178	1	0
8	0.922117	0.000661812	0.000661812	0.5	0
9	0.922117	0.000710043	0.000710043	1	0
10	0.922117	4.03605e-05	4.03605e-05	1	1
11	0.922117	6.21871e-07	6.21871e-07	1	0
12	0.922117	2.03648e-06	2.03648e-06	1	0
13	0.922117	1.5208e-10	1.5208e-10	1	0
iterration: 252


...

0	377.114	25.9759	25.9759	1	1
1	373.193	5.495	5.495	1	1
2	372.87	0.973051	0.973051	1	1
3	372.851	0.350649	0.350649	1	1
4	372.845	0.086141	0.086141	1	1
5	372.842	0.0517046	0.0517046	1	1
6	372.842	0.0257803	0.0257803	1	1
7	372.842	0.00825707	0.00825707	1	1
8	372.842	0.00265285	0.00265285	1	1
9	372.842	0.00133571	0.00133571	1	1
10	372.842	0.000862741	0.000862741	1	1
11	372.842	0.000585521	0.000585521	1	1
12	372.842	0.000451114	0.000451114	1	1
13	372.842	0.000499077	0.000499077	1	1
14	372.842	0.000388287	0.000388287	1	1
15	372.842	0.000161223	0.000161223	1	1
16	372.842	7.61968e-05	7.61968e-05	1	1
17	372.842	4.00382e-05	4.00382e-05	1	1
18	372.842	2.35127e-05	2.35127e-05	1	1
19	372.842	1.54771e-05	1.54771e-05	1	1
20	372.842	1.13005e-05	1.13005e-05	1	1
21	372.842	8.94929e-06	8.94929e-06	1	1
22	372.842	7.49507e-06	7.49507e-06	1	1
23	372.842	6.51495e-06	6.51495e-06	1	1
24	372.842	5.80906e-06	5.80906e-06	1	1
25	372.842	5.27515e-06	5.27515e-06	1	1
26	372.842	4.85774e-06	4.85774e-06	1	1
27	372.842

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	392.56	25.992	25.992	1	1
1	388.615	6.12753	6.12753	1	1
2	388.273	1.10874	1.10874	1	1
3	388.253	0.356992	0.356992	1	1
4	388.247	0.0884367	0.0884367	1	1
5	388.244	0.0556407	0.0556407	1	1
6	388.243	0.0291972	0.0291972	1	1
7	388.243	0.00960427	0.00960427	1	1
8	388.243	0.00294088	0.00294088	1	1
9	388.243	0.00140786	0.00140786	1	1
10	388.243	0.00089871	0.00089871	1	1
11	388.243	0.000606249	0.000606249	1	1
12	388.243	0.000448184	0.000448184	1	1
13	388.243	0.000480085	0.000480085	1	1
14	388.243	0.000375274	0.000375274	1	1
15	388.243	0.000156071	0.000156071	1	1
16	388.243	7.4315e-05	7.4315e-05	1	1
17	388.243	3.94371e-05	3.94371e-05	1	1
18	388.243	2.33768e-05	2.33768e-05	1	1
19	388.243	1.54195e-05	1.54195e-05	1	1
20	388.243	1.11458e-05	1.11458e-05	1	1
21	388.243	8.62936e-06	8.62936e-06	1	1
22	388.243	6.99885e-06	6.99885e-06	1	1
23	388.243	5.85749e-06	5.85749e-06	1	1
24	388.243	5.00392e-06	5.00392e-06	1	1
25	388.243	4.34048e-06	4.34048e-06	1	1
26	388.243	3.80789e-06	3.80789e-06	1	1
27	388.243	3

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	408.324	26.0307	26.0307	1	1
1	404.355	6.69033	6.69033	1	1
2	403.994	1.25479	1.25479	1	1
3	403.972	0.353852	0.353852	1	1
4	403.966	0.090648	0.090648	1	1
5	403.962	0.0595307	0.0595307	1	1
6	403.962	0.0328961	0.0328961	1	1
7	403.961	0.0111715	0.0111715	1	1
8	403.961	0.00329028	0.00329028	1	1
9	403.961	0.00148817	0.00148817	1	1
10	403.961	0.000935808	0.000935808	1	1
11	403.961	0.000629409	0.000629409	1	1
12	403.961	0.000451129	0.000451129	1	1
13	403.961	0.000468812	0.000468812	1	1
14	403.961	0.000371478	0.000371478	1	0
15	403.961	0.0145859	0.0145859	1	1
16	403.961	1.18059e-05	1.18059e-05	1	1
Computing negative curvature direction for scaled tau = 7.95776e-10
Spectra unsuccessful after 21 iterations
Negative curvature direction calculation failed
17	403.961	6.57634e-07	6.57634e-07	1	1
Computing negative curvature direction for scaled tau = 7.95776e-10
Spectra unsuccessful after 21 iterations
Negative curvature direction calculation failed
18	403.961	4.81288e-07	4.81288e-07	1	1
Computing n

In [ ]:
from helpers import write_obj
file = '../data/NoCollision/' + knot_name
write_obj(file, rod_list)

In [ ]:
# Load the centerline from file...
file = '../data/NoCollision/' + knot_name
knot = read_nodes_from_file(file)
rod_radius = 0.2
material = elastic_rods.RodMaterial('ellipse', 2000, 0.3, [rod_radius, rod_radius])
pr = define_periodic_rod(knot[::1], material)
rod_list = elastic_knots.PeriodicRodList([pr])

In [ ]:
view = Viewer(rod_list, width=1024, height=800)
view.show()

In [ ]:
from helpers import write_obj
file = '../data/NoCollision/reduced' + knot_name
write_obj(file, rod_list)